# AGENTIC AI
## SESSION 19

Assignment questions and answers


## Question 1

Install the GitHub MCP Server on your local machine by following the official documentation and verify the installation by running the server and accessing its default endpoint.

**Answer:**

**1. Install and Run the Server**

Set your GitHub personal access token and run the official server using Docker or `npx`:

*Using Docker:*
```bash
docker run -i --rm -e GITHUB_PERSONAL_ACCESS_TOKEN=<YOUR_TOKEN> mcp/github
```

*Using npx:*
```bash
GITHUB_PERSONAL_ACCESS_TOKEN=<YOUR_TOKEN> npx -y @modelcontextprotocol/server-github
```

**2. Verify the Installation**

The GitHub MCP Server operates over standard input/output (`stdio`) by default, rather than an HTTP endpoint. 

To verify it is running and accessible, send a standard JSON-RPC initialization request via the command line:

```bash
echo '{"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2024-11-05", "capabilities": {}, "clientInfo": {"name": "test", "version": "1.0"}}}' | GITHUB_PERSONAL_ACCESS_TOKEN=<YOUR_TOKEN> npx -y @modelcontextprotocol/server-github
```

**Verification Success:**  
The server outputs a JSON response containing `"serverInfo": {"name": "github-mcp-server"}` and the server's supported capabilities.

## Question 2

Create a new GitHub repository named 'playlist-manager', clone it to your system, and push a README.md file describing a simple Spotify-style playlist feature you would like to build.

**Answer:**

1. **Create the repository on GitHub:**
   * Go to GitHub, click **New repository**, name it `playlist-manager`, keep it public/private, leave "Initialize with a README" unchecked, and click **Create repository**.

2. **Clone, create README, and push (run in your terminal):**

```bash
git clone https://github.com/<your-username>/playlist-manager.git
cd playlist-manager

echo "# Playlist Manager

A Spotify-style feature that allows users to create custom playlists, search for tracks by title or artist, add or remove songs, and reorder the tracklist using drag-and-drop." > README.md

git add README.md
git commit -m "Add README describing playlist feature"
git branch -M main
git push -u origin main
```

## Question 3

Configure the GitHub MCP Server to automatically trigger a message in your terminal whenever a new issue is created in your 'playlist-manager' repository.

Hint: Use the webhook integration feature of the MCP Server and test by creating a dummy issue on GitHub.

**Answer:**

1. **Start the GitHub MCP Server with Webhook Support:**
   Run the MCP server exposing a local webhook port (e.g., port 8000) using your GitHub Personal Access Token:
   ```bash
   env GITHUB_PERSONAL_ACCESS_TOKEN="your_token" npx -y @modelcontextprotocol/server-github --webhook-port 8000
   ```

2. **Expose Localhost to the Internet:**
   Use a tunneling tool like `ngrok` so GitHub can reach your local MCP server:
   ```bash
   ngrok http 8000
   ```
   *Copy the generated forwarding URL (e.g., `https://<unique-id>.ngrok-free.app`).*

3. **Configure the Webhook in GitHub:**
   - Go to your `playlist-manager` repository on GitHub.
   - Navigate to **Settings** > **Webhooks** > **Add webhook**.
   - Set **Payload URL** to: `https://<unique-id>.ngrok-free.app/webhook`
   - Set **Content type** to: `application/json`.
   - Under **Which events would you like to trigger this webhook?**, select **Let me select individual events** and check **Issues**.
   - Click **Add webhook**.

4. **Test the Trigger:**
   - Create a dummy issue in the `playlist-manager` repository.
   - Verify the notification message appears in your terminal running the MCP server.

## Question 4

Modify the MCP Server configuration to only respond to pull request events labeled 'bug' in your repository, and print the PR title and author to the console.

**Answer:**

```json
{
  "mcpServers": {
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": {
        "GITHUB_PERSONAL_ACCESS_TOKEN": "<YOUR_TOKEN>"
      }
    }
  }
}
```

```python
# Event handler logic for the MCP-connected service:
def handle_webhook(event):
    if event.get("action") in ["opened", "labeled"]:
        labels = [l["name"] for l in event["pull_request"]["labels"]]
        if "bug" in labels:
            title = event["pull_request"]["title"]
            author = event["pull_request"]["user"]["login"]
            print(f"Title: {title}, Author: {author}")
```

## Question 5

Use ChatGPT or GitHub Copilot to generate a sample webhook handler function for the MCP Server that posts a thank you comment on any new issue in your repository, then implement and test it.

**Answer:**

Here is the implementation of the webhook handler using Python, FastMCP, and the GitHub API:

### 1. Implementation (`server.py`)

```python
import os
import requests
from fastmcp import FastMCP

mcp = FastMCP("github-webhook-server")
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

@mcp.tool()
def handle_github_issue_webhook(event_type: str, payload: dict) -> str:
    """Handles GitHub webhook events and comments on newly opened issues."""
    if event_type != "issues" or payload.get("action") != "opened":
        return "Ignored: Not a newly opened issue event."

    issue = payload.get("issue", {})
    comments_url = issue.get("comments_url")
    user = issue.get("user", {}).get("login", "there")

    if not comments_url:
        return "Error: comments_url not found in payload."

    headers = {
        "Authorization": f"Bearer {GITHUB_TOKEN}",
        "Accept": "application/vnd.github+json",
    }
    body = {"body": f"Hi @{user}, thank you for opening this issue!"}

    response = requests.post(comments_url, json=body, headers=headers)
    
    if response.status_code == 201:
        return "Success: Thank you comment posted."
    return f"Failed: {response.status_code} - {response.text}"

if __name__ == "__main__":
    mcp.run()
```

### 2. Testing

Run this test script locally to verify the function using a mock GitHub payload:

```python
import os
from server import handle_github_issue_webhook

# Set your personal access token and target a real issue for live testing
os.environ["GITHUB_TOKEN"] = "ghp_your_actual_github_token"

mock_payload = {
    "action": "opened",
    "issue": {
        "user": {"login": "octocat"},
        # Replace with your repo's test issue comments URL
        "comments_url": "https://api.github.com/repos/YOUR_USERNAME/YOUR_REPO/issues/1/comments"
    }
}

result = handle_github_issue_webhook(event_type="issues", payload=mock_payload)
print(result)
```